## 4 - Data Validation Generico DAG

Descubre el grafo causal desde los datos: muestra temporal (t−1, t), FCI con restricciones temporales, DirectLiNGAM como confirmación independiente y clasificación de roles por variable.

In [ ]:
pip install causal-learn

# Parámetros

Único bloque a editar por embotellador-país.

In [ ]:
BU = 'MEX'

In [ ]:
# Muestreo del discovery: PDVs y filas para las pruebas de independencia
N_PDV_FCI = 60_000
ROWS_FCI  = 120_000
ALPHA_FCI = 1e-5
SEED_FCI  = 42

# Setup y Lectura de Datos

In [ ]:
import pyspark.sql.functions as sf
import pandas as pd
import numpy as np
import time
import json
import matplotlib.pyplot as plt
from causallearn.utils.PCUtils.BackgroundKnowledge import BackgroundKnowledge
from causallearn.graph.GraphNode import GraphNode
from causallearn.search.ConstraintBased.FCI import fci
from causallearn.search.FCMBased import lingam
from matplotlib.patches import FancyArrowPatch, FancyBboxPatch
from matplotlib.lines import Line2D

ruta_panel = f"abfss://{container}@{storageAccount}.dfs.core.windows.net/CTG/{BU}/PanelModelo/parquet/"
ruta_decisiones = f"abfss://{container}@{storageAccount}.dfs.core.windows.net/CTG/{BU}/Modelo/capacidades"

# Capacidades del modelo desde el archivo de decisiones
rows = spark.read.text(ruta_decisiones).collect()
DECISIONES = json.loads('\n'.join(r[0] for r in rows))
CAPACIDADES = list(DECISIONES['formas'].keys())

print(f"BU: {BU}")
print(f"Capacidades: {CAPACIDADES}")
print(f"Universo: {DECISIONES.get('universo')} | escala: {DECISIONES.get('convencion_proporciones')}")

# Muestra temporal (t−1, t)

In [ ]:
# Variables en t-1 y t (capacidades + mediadores frecuencia/volumen + venta) + estáticos del PDV
panel_cols = spark.read.parquet(ruta_panel).columns
CAPS_V2 = [c for c in CAPACIDADES if c in panel_cols]
cols_v2 = ['id_cliente','period_id','ingreso_neto_core_real','tamano_cliente','canal',
           'ordenes_promedio','unit_cases_total'] + CAPS_V2
ids_fci = (spark.read.parquet(ruta_panel).select('id_cliente').distinct()
           .orderBy(sf.rand(seed=SEED_FCI)).limit(N_PDV_FCI))
pdf = (spark.read.parquet(ruta_panel).select(*cols_v2)
       .join(ids_fci,'id_cliente','inner').toPandas())
pdf = pdf[pdf['ingreso_neto_core_real'].notna()].copy()
pdf['log_y'] = np.log(pdf['ingreso_neto_core_real'].clip(lower=0.01))
pdf['freq']  = pdf['ordenes_promedio'].fillna(0)
pdf['vol']   = np.log(pdf['unit_cases_total'].clip(lower=0.01))
pdf['tam']   = pdf['tamano_cliente'].map({'MICRO':1,'CHICO':1,'Sin Asignar':1,'MEDIANO':2,'GRANDE':3,'EXT-GDE':3}).fillna(1)
pdf['tenure']= pdf.groupby('id_cliente')['period_id'].transform('nunique')

# Dummies de canal: solo las categorías con variación (k-1 dummies, máximo 3)
_vc = pdf['canal'].value_counts()
CANAL_DUMMIES = []
for _i,_cn in enumerate(_vc.index[:min(3, len(_vc)-1)], 1):
    pdf[f'canal{_i}'] = (pdf['canal']==_cn).astype(float)
    CANAL_DUMMIES.append(f'canal{_i}')

DYN_V2 = CAPS_V2 + ['freq','vol','log_y']
for c in DYN_V2:
    _b = pdf.set_index(['id_cliente','period_id'])[c]
    _ix = pd.MultiIndex.from_arrays([pdf['id_cliente'], pdf['period_id']-1])
    pdf[f'{c}_L'] = _b.reindex(_ix).values
d_v2 = pdf.dropna(subset=[f'{c}_L' for c in DYN_V2]).copy()
EST_V2 = ['tam','tenure'] + CANAL_DUMMIES
NODOS_V2 = EST_V2 + [f'{c}_L' for c in DYN_V2] + DYN_V2
X_v2 = d_v2[NODOS_V2].to_numpy(np.float64)
X_v2 = (X_v2 - X_v2.mean(0)) / np.where(X_v2.std(0)==0, 1, X_v2.std(0))
_rng = np.random.RandomState(SEED_FCI)
Xs_v2 = X_v2[_rng.choice(len(X_v2), size=min(ROWS_FCI,len(X_v2)), replace=False)]
print(f"DAG: {len(NODOS_V2)} nodos | {len(Xs_v2):,} filas | caps: {CAPS_V2} | canal dummies: {CANAL_DUMMIES}")

# FCI con restricciones temporales

In [ ]:
# Restricciones: nada causa los estáticos; el presente no causa el pasado; venta(t) es sumidero
_nodes = [GraphNode(f'X{i+1}') for i in range(len(NODOS_V2))]
_nm = {NODOS_V2[i]: _nodes[i] for i in range(len(NODOS_V2))}
T1_V2 = [f'{c}_L' for c in DYN_V2]; T2_V2 = DYN_V2
bk_v2 = BackgroundKnowledge()
for a in T1_V2 + T2_V2:
    for e in EST_V2: bk_v2.add_forbidden_by_node(_nm[a], _nm[e])
for a in T2_V2:
    for b in T1_V2: bk_v2.add_forbidden_by_node(_nm[a], _nm[b])
for b in NODOS_V2:
    if b != 'log_y': bk_v2.add_forbidden_by_node(_nm['log_y'], _nm[b])
_t0 = time.time()
g_v2, _ = fci(Xs_v2, independence_test_method='fisherz', alpha=ALPHA_FCI,
              background_knowledge=bk_v2, node_names=[f'X{i+1}' for i in range(len(NODOS_V2))], verbose=False)
_lab = {f'X{i+1}': NODOS_V2[i] for i in range(len(NODOS_V2))}
FCIE_V2 = [{'a':_lab[e.get_node1().get_name()],'b':_lab[e.get_node2().get_name()],
            'ea':str(e.get_endpoint1()),'eb':str(e.get_endpoint2())} for e in g_v2.get_graph_edges()]
DIR_FCI = set((e['a'],e['b']) for e in FCIE_V2 if e['ea']=='TAIL' and e['eb']=='ARROW')
print(f"FCI temporal: {len(FCIE_V2)} aristas ({len(DIR_FCI)} dirigidas) en {time.time()-_t0:.0f}s")

# DirectLiNGAM e intersección

In [ ]:
# Segundo algoritmo (no-gaussianidad): las aristas donde ambos coinciden son de alta confianza
_t0 = time.time()
_ml = lingam.DirectLiNGAM(); _ml.fit(Xs_v2)
_ADJ = _ml.adjacency_matrix_   # ADJ[i,j]!=0 => j -> i
LING_V2 = set()
for i in range(len(NODOS_V2)):
    for j in range(len(NODOS_V2)):
        if abs(_ADJ[i,j]) > 0.01: LING_V2.add((NODOS_V2[j], NODOS_V2[i]))
def _temporal_ok(a,b):
    if b in EST_V2: return False
    if a in T2_V2 and b in T1_V2: return False
    if a=='log_y' and b!='log_y': return False
    return True
LING_V2 = {(a,b) for a,b in LING_V2 if _temporal_ok(a,b)}
AMBAS_V2 = DIR_FCI & LING_V2
print(f"DirectLiNGAM: {len(LING_V2)} dirigidas ({time.time()-_t0:.0f}s) | COINCIDEN con FCI: {len(AMBAS_V2)}")
for a,b in sorted(AMBAS_V2): print(f"   {a} → {b}")

# Clasificación de roles y decisión de especificación

In [ ]:
# Rol de cada variable según el grafo: mediador / confusor / tratamiento
_adj = {}
for e in FCIE_V2:
    _adj.setdefault(e['a'],set()).add(e['b']); _adj.setdefault(e['b'],set()).add(e['a'])
E_ALL_V2 = DIR_FCI | LING_V2
print(f"{'variable':<18}{'rol descubierto':<52}{'decisión'}")
for base in ['freq','vol']:
    _porcap = any((c,base) in E_ALL_V2 or (f'{c}_L',base) in E_ALL_V2 for c in CAPS_V2)
    _ay = (base,'log_y') in E_ALL_V2 or (f'{base}_L','log_y') in E_ALL_V2 or ('log_y' in _adj.get(base,set()))
    _rol = f'MEDIADOR (cap → {base} → venta)' if (_porcap and _ay) else ('asociado a venta' if _ay else 'sin camino a venta')
    _dec = 'EXCLUIR de X (bad control)' if 'MEDIADOR' in _rol else 'no entra'
    print(f"{base:<18}{_rol:<52}{_dec}")
for e0 in EST_V2:
    _ay = ('log_y' in _adj.get(e0,set())) or (e0,'log_y') in E_ALL_V2
    _ac = any(c in _adj.get(e0,set()) or f'{c}_L' in _adj.get(e0,set()) for c in CAPS_V2)
    _rol = 'CONFUSOR (→adopción y →venta)' if (_ay and _ac) else ('solo→venta (no sesga omitirlo)' if _ay else ('solo→adopción' if _ac else 'aislado'))
    _dec = 'CONTROLAR (grupo/Mundlak)' if 'CONFUSOR' in _rol else ('control opcional' if (_ay or _ac) else 'no necesario')
    print(f"{e0:<18}{_rol:<52}{_dec}")
for c in CAPS_V2:
    _dy = (c,'log_y') in E_ALL_V2 or (f'{c}_L','log_y') in E_ALL_V2
    _as = 'log_y' in _adj.get(c,set()) or 'log_y' in _adj.get(f'{c}_L',set())
    _rol = 'CAUSAL → venta (dirigida)' if _dy else ('asociada (dirección vía modelo)' if _as else 'sin arista directa (vía combos)')
    print(f"{c:<18}{_rol:<52}ENTRA en X (tratamiento)")

# Gráfico por capas

In [ ]:
_ORDER = CAPS_V2 + ['freq','vol']
_NICE = {'tam':'Tamaño','tenure':'Antigüedad','canal1':'Canal 1','canal2':'Canal 2','canal3':'Canal 3',
         'freq':'Frecuencia','vol':'Volumen','log_y':'VENTA CORE','Multicategory':'Multicategoría',
         'PedidoSugerido':'Pedido Sugerido','GuidedMissions':'Guided Missions','DigitalServices':'Digital Services'}
def _nice(n):
    b = n[:-2] if n.endswith('_L') else n
    return _NICE.get(b,b) + ('  (t−1)' if n.endswith('_L') else '')
def _style(n):
    b = n[:-2] if n.endswith('_L') else n
    if b=='log_y': return ('#7B1F1F','#FBE9E9')
    if b in ('freq','vol'): return ('#8a6d00','#FFF6DC')
    if b in EST_V2: return ('#2E5B1E','#E8F3E0')
    return ('#5B7FBF','#EEF3FB') if n.endswith('_L') else ('#1F3864','#DCE6F6')
_pos = {}
for i,c in enumerate(_ORDER):
    _y = 0.93 - i*0.088; _pos[f'{c}_L']=(0.10,_y); _pos[c]=(0.52,_y)
_pos['log_y_L']=(0.10,0.93-len(_ORDER)*0.088); _pos['log_y']=(0.90,0.50)
for i,n in enumerate(EST_V2): _pos[n]=(0.10+i*0.17,-0.055)
fig,ax = plt.subplots(figsize=(15,8.6)); ax.axis('off'); ax.set_xlim(0,1); ax.set_ylim(-0.13,1.04)
for x0,x1,t in [(0.015,0.235,'PASADO (t−1)'),(0.40,0.665,'PRESENTE (t)'),(0.79,0.995,'OUTCOME')]:
    ax.add_patch(FancyBboxPatch((x0,0.005),x1-x0,0.995,boxstyle='round,pad=0.008',fc='#F7F8FA',ec='#D9D9D9',lw=1,zorder=0))
    ax.text((x0+x1)/2,1.015,t,ha='center',fontsize=9.5,weight='bold',color='#7F7F7F')
ax.text(0.5,-0.115,'ATRIBUTOS ESTÁTICOS (confusores potenciales)',ha='center',fontsize=9,weight='bold',color='#2E5B1E')
def _rad(a,b):
    (x1,y1),(x2,y2)=_pos[a],_pos[b]
    return 0.35 if abs(x1-x2)<0.05 else (0.12 if (y1+y2)/2>0.5 else -0.12)
for e in FCIE_V2:
    a,b=e['a'],e['b']; _d1=(e['ea']=='TAIL' and e['eb']=='ARROW'); _cf=(a,b) in AMBAS_V2
    if _cf: _c,_lw,_ls,_al,_z='#1A7A3A',2.6,'-',1.0,3
    elif _d1: _c,_lw,_ls,_al,_z='#404040',1.4,'-',0.9,2
    else: _c,_lw,_ls,_al,_z='#C8A46E',0.8,(0,(4,3)),0.45,1
    ax.add_patch(FancyArrowPatch(_pos[a],_pos[b],connectionstyle=f'arc3,rad={_rad(a,b)}',
        arrowstyle='-|>' if (_d1 or _cf) else '<|-|>',mutation_scale=11,lw=_lw,color=_c,ls=_ls,alpha=_al,zorder=_z,shrinkA=17,shrinkB=17))
for n,(x,y) in _pos.items():
    _ec,_fc=_style(n); _big=(n=='log_y')
    ax.annotate(_nice(n),(x,y),ha='center',va='center',fontsize=9.5 if _big else 7.6,weight='bold',color=_ec,zorder=5,
        bbox=dict(boxstyle='round,pad=0.42' if _big else 'round,pad=0.32',fc=_fc,ec=_ec,lw=2.2 if _big else 1.5))
ax.legend(handles=[Line2D([0],[0],color='#1A7A3A',lw=2.6,label='Dirigida — confirmada por FCI y DirectLiNGAM'),
                   Line2D([0],[0],color='#404040',lw=1.4,label='Dirigida — FCI (orientada por el tiempo)'),
                   Line2D([0],[0],color='#C8A46E',lw=0.9,ls='--',label='Asociación — sin resolver / confusor latente')],
          loc='upper right',bbox_to_anchor=(0.998,0.985),fontsize=8.4,framealpha=0.95)
ax.set_title(f'DAG CATALYST — FCI temporal + DirectLiNGAM ({BU} · {len(NODOS_V2)} nodos · α={ALPHA_FCI})',fontsize=13,weight='bold',color='#1F3864',pad=14)
plt.tight_layout(); plt.show()